In [1]:
!pip install langchain langchain-community langchain-google-genai faiss-cpu sentence-transformers

In [2]:
import os
import google.generativeai as genai

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_google_genai import ChatGoogleGenerativeAI

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
# -----------------------------
# 1. Configure Gemini API
# -----------------------------

# os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
# genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [4]:
# -----------------------------
# 2. Sample Knowledge Base
# -----------------------------

documents = [

Document(page_content="""Tesla produces electric vehicles and energy storage systems.
The company focuses on sustainable transportation and was led by Elon Musk."""),

Document(page_content="""Toyota is a Japanese automobile manufacturer known for hybrid
vehicles such as the Prius and for pioneering hybrid technology."""),

Document(page_content="""BMW is a German company producing luxury vehicles and motorcycles.
It is known for engineering performance and premium design."""),

Document(page_content="""Electric vehicles reduce carbon emissions compared to gasoline
vehicles and rely on battery-powered electric motors.""")

]


# -----------------------------
# 3. Document Chunking
# -----------------------------

splitter = RecursiveCharacterTextSplitter(
chunk_size=200,
chunk_overlap=50
)

docs = splitter.split_documents(documents)


# -----------------------------
# 4. Embedding Model
# -----------------------------

embedding_model = HuggingFaceEmbeddings(
model_name="sentence-transformers/all-MiniLM-L6-v2"
)


/tmp/ipykernel_1448/2352944007.py:38: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# -----------------------------
# 5. Persistent Vector DB
# -----------------------------

db_path = "vector_store"

if os.path.exists(db_path):
  vector_db = FAISS.load_local(db_path,
  embedding_model,
  allow_dangerous_deserialization=True
)

else:
  vector_db = FAISS.from_documents(docs, embedding_model)
  vector_db.save_local(db_path)

retriever = vector_db.as_retriever(search_kwargs={"k":4})



In [7]:
# -----------------------------
# 6. Gemini LLM
# -----------------------------

llm = ChatGoogleGenerativeAI(
model="gemini-2.5-flash",
temperature=0
)


# -----------------------------
# 7. Multi-Query Generation
# -----------------------------

multi_query_prompt = ChatPromptTemplate.from_template(
"""
Generate 3 alternative search queries for the question.

Question:
{question}

Return each query on a new line.
"""
)


multi_query_chain = multi_query_prompt | llm


def generate_queries(question):
  result = multi_query_chain.invoke({"question": question})
  queries = result.content.split("\n")
  queries.append(question)
  return queries


# -----------------------------
# 8. Retrieve Documents
# -----------------------------

def retrieve_documents(question):
  queries = generate_queries(question)
  results = []
  for q in queries:
    docs = retriever.invoke(q)
    results.extend(docs)

  return results


# -----------------------------
# 9. Simple Reranking
# -----------------------------

def rerank_docs(docs):
  unique = {d.page_content: d for d in docs}
  return list(unique.values())[:3]


# -----------------------------
# 10. Format Context
# -----------------------------

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)


In [8]:
# -----------------------------
# 11. Final Answer Prompt
# -----------------------------

answer_prompt = ChatPromptTemplate.from_template(
"""
You are an AI assistant answering questions using retrieved knowledge.

Context:
{context}

Question:
{question}

Provide a clear answer using the context.
"""
)


# -----------------------------
# 12. Full LCEL Pipeline
# -----------------------------

rag_chain = (

{
"context": RunnableLambda(
lambda q: format_docs(
rerank_docs(
retrieve_documents(q)
)
)
),
"question": RunnablePassthrough()
}

| answer_prompt
| llm
)




In [9]:
# -----------------------------
# 13. Interactive Loop
# -----------------------------

while True:
  question = input("\nAsk a question (type 'exit'): ")
  if question.lower() == "exit":
    break
  response = rag_chain.invoke(question)
  print("\nAnswer:\n")
  print(response.content)


Ask a question (type 'exit'): what are the benefits of EVs?

Answer:

Electric vehicles (EVs) reduce carbon emissions compared to gasoline vehicles. This contributes to sustainable transportation.

Ask a question (type 'exit'): exit
